In [1]:
import pandas as pd 
import numpy as np 
from sklearn.model_selection import train_test_split, KFold, cross_val_score 
from sklearn.preprocessing   import StandardScaler 
from sklearn.linear_model    import Ridge 
from sklearn.ensemble        import RandomForestRegressor, GradientBoostingRegressor 
from sklearn.metrics         import mean_absolute_error, r2_score, mean_squared_error 
 
df = pd.read_parquet('../data/features.parquet') 
 
# --- Целевой признак --- 
df['log_views'] = np.log1p(df['view_count']) 
 
# --- Признаки БЕЗ data leakage --- 
# Используем только то, что известно до/в момент публикации 
BASE_FEATURES = [ 
    'duration_sec', 'log_duration', 
    'title_length', 'title_word_count', 'title_upper_ratio', 
    'has_tags',     'tag_count', 
    'publish_hour', 'publish_dayofweek', 
] 
# Признаки из API (если mock — будут нули) 
API_FEATURES = ['log_subscribers', 'log_channel_videos', 'log_channel_total_views'] 
 
# One-Hot Encoding категорий 
df_model = pd.get_dummies(df, columns=['category'], drop_first=False) 
cat_features = [c for c in df_model.columns if c.startswith('category_')] 
 
FEATURES = BASE_FEATURES + API_FEATURES + cat_features 
FEATURES = [f for f in FEATURES if f in df_model.columns] 
 
X = df_model[FEATURES].fillna(0) 
y = df_model['log_views'] 
 
print(f'Признаков: {len(FEATURES)}, Строк: {len(X)}') 
print(f'Признаки: {FEATURES}') 

Признаков: 17, Строк: 28
Признаки: ['duration_sec', 'log_duration', 'title_length', 'title_word_count', 'title_upper_ratio', 'has_tags', 'tag_count', 'publish_hour', 'publish_dayofweek', 'log_subscribers', 'log_channel_videos', 'log_channel_total_views', 'category_id', 'category_Film & Entertainment', 'category_Gaming', 'category_Music', 'category_Other']


In [2]:
X_train, X_test, y_train, y_test = train_test_split( 
    X, y, test_size=0.2, random_state=42 
) 
print(f'Train: {len(X_train)} строк  |  Test: {len(X_test)} строк') 
# При 376 строках: ~301 train, 75 test 
 
# Масштабирование (fit ТОЛЬКО на train — никогда не на всём датасете!) 
scaler = StandardScaler() 
X_train_sc = scaler.fit_transform(X_train) 
X_test_sc  = scaler.transform(X_test)   # transform без fit

Train: 22 строк  |  Test: 6 строк


In [3]:
# KFold(5) делит 301 train-строку на 5 фолдов по ~60 строк. 
cv = KFold(n_splits=5, shuffle=True, random_state=42) 
 
models = { 
    'Ridge (L2)':          Ridge(alpha=1.0), 
    'RandomForest':        RandomForestRegressor(n_estimators=100, random_state=42), 
    'GradientBoosting':    GradientBoostingRegressor(n_estimators=100, random_state=42), 
} 
 
results = {} 
for name, model in models.items(): 
    # Ridge требует масштабирования, деревья — нет 
    X_tr = X_train_sc if 'Ridge' in name else X_train 
 
    # Кросс-валидация: отрицательное MSE (sklearn минимизирует, поэтому '-') 
    cv_neg_mse = cross_val_score(model, X_tr, y_train, 
                                  cv=cv, scoring='neg_mean_squared_error') 
    cv_rmse = np.sqrt(-cv_neg_mse).mean() 
    cv_r2   = cross_val_score(model, X_tr, y_train, cv=cv, scoring='r2').mean() 
 
    # Обучение на полном train и оценка на test 
    model.fit(X_tr, y_train) 
    X_te = X_test_sc if 'Ridge' in name else X_test 
    y_pred = model.predict(X_te) 
 
    test_mae  = mean_absolute_error(y_test, y_pred) 
    test_r2   = r2_score(y_test, y_pred) 
 
    results[name] = { 
        'CV RMSE (log)':  round(cv_rmse, 4), 
        'CV R2':          round(cv_r2, 4), 
        'Test MAE (log)': round(test_mae, 4), 
        'Test R2':        round(test_r2, 4), 
    } 
 
print(pd.DataFrame(results).T)

                  CV RMSE (log)   CV R2  Test MAE (log)  Test R2
Ridge (L2)               1.3759 -0.8846          1.1960  -0.8922
RandomForest             1.1818 -0.1808          0.9999  -0.2317
GradientBoosting         1.2583 -0.4289          1.4990  -1.7630


In [4]:
import joblib
import os
from sklearn.ensemble import RandomForestRegressor

# 1. Автоматически создаем папку models, если её ещё нет
os.makedirs('../models', exist_ok=True)

# 2. Создаем чистый RandomForestRegressor с вашими параметрами
rf_best_model = RandomForestRegressor(n_estimators=100, random_state=42)

# 3. Обучаем его на ваших тренировочных данных
# (Переменные X_tr и y_train должны быть в памяти, если запустили ячейки выше)
rf_best_model.fit(X_tr, y_train)

# 4. Сохраняем модель в файл
joblib.dump(rf_best_model, '../models/youtube_rf_model.pkl')

print("Модель RandomForest успешно обучена и сохранена в файл youtube_rf_model.pkl!")


Модель RandomForest успешно обучена и сохранена в файл youtube_rf_model.pkl!


In [6]:
import joblib
import os

# 1. Создаем папку models в корне проекта
os.makedirs('../models', exist_ok=True)

# 2. Вытаскиваем и обучаем лучшую модель (RandomForest) из вашего словаря models
best_model = models['RandomForest']
best_model.fit(X_tr, y_train)

# 3. Сохраняем саму модель
joblib.dump(best_model, '../models/model.joblib')

# 4. Сохраняем список признаков (колонок), которые модель ждет на вход
# Это критически важно, чтобы наш будущий API-сервер знал порядок колонок
FEATURES = list(X_tr.columns)
joblib.dump(FEATURES, '../models/features.joblib')

print("Все файлы успешно созданы в папке models/!")


Все файлы успешно созданы в папке models/!
